# GNN Fraud Detection - Cloud GPU Training Pipeline
Because local PCs without a dedicated GPU can take hours to train PyTorch Geometric models, we use Google Colab to offload the heavy lifting for free. Once trained, we will download the "brain" back to our local PC for lightning-fast CPU inference.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

**Instructions:**
1. On your local PC, zip up your entire `ml` folder into a file called `ml.zip`.
2. Upload `ml.zip` to the root of your Google Drive.
3. Run the cell below to unpack it into this cloud machine.

In [ ]:
# 2. Unzip your uploaded project
!unzip -q /content/drive/MyDrive/ml.zip -d /content/

In [ ]:
# 3. Install required deep learning libraries exactly as specified in our manifest
!pip install -r /content/ml/requirements.txt
import torch
print('GPU Available for ultra-fast training:', torch.cuda.is_available())

In [ ]:
# 4. Secure Kaggle Authentication for Data Download
import os
from getpass import getpass

print("Please provide your Kaggle API credentials to securely download the 4GB dataset.")
os.environ['KAGGLE_USERNAME'] = input('Enter your Kaggle Username: ')
os.environ['KAGGLE_KEY'] = getpass('Enter your Kaggle API Key: ')
print("\nCredentials injected securely! Triggering download script...")

!python /content/ml/data/download_data.py

In [ ]:
# 5. Run the Master Orchestrator!
# This single command triggers System 1 (Data), System 2 (Baseline), System 3 (GNN), System 4 (XAI) and Triton Export.
!python /content/ml/train_pipeline.py

In [ ]:
# 6. Zip and Download the Trained Brain!
import shutil
import os
from google.colab import files

# Stage the ONNX models into the artifacts folder so they get zipped together
triton_source = '/content/ml/export/triton_model_repository'
triton_dest = '/content/ml/models/artifacts/triton_model_repository'

if os.path.exists(triton_source):
    if os.path.exists(triton_dest):
        shutil.rmtree(triton_dest)
    shutil.copytree(triton_source, triton_dest)
    print("Triton ONNX deployment models successfully staged for download.")

# Zip the unified artifacts folder (PyTorch Weights, Scalers, ONNX models, Redis JSON)
shutil.make_archive('/content/trained_artifacts', 'zip', '/content/ml/models/artifacts')

# Trigger the download to your local PC
files.download('/content/trained_artifacts.zip')
print('Download complete! Extract this zip directly over your local ml/models/artifacts/ folder.')